## Planning:
- Load data
- Create sample FE - sum of boolean and proportion of boolean and Digit Position
- Create TE in run.cv
- Implement for XGBoost

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [4]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [5]:
train = pd.read_csv(competition_path/"data/train.csv")
test = pd.read_csv(competition_path/"data/test.csv")
submit = pd.read_csv(competition_path/"data/sample_submission.csv")

In [6]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()
numeric_cols_ex = numeric_cols.copy()
numeric_cols_ex.remove("is_fraud")

In [7]:
bool_cols = train.select_dtypes(include='bool').columns.tolist()

In [8]:
obj_cols = train.select_dtypes(include='object').columns.tolist()

In [9]:
# checking the distribution on each of category in categorical variable
for col in obj_cols:
    counts = train.groupby(col)["id"].count()
    proportion = counts/counts.sum()

    print(f'col: {col}')
    summary = pd.DataFrame({'count': counts, 'proportion': proportion})
    print(summary)
    print('-'*20)

col: merchant_category
                    count  proportion
merchant_category                    
Crypto Exchange      5285    0.007550
Electronics         48693    0.069565
Fuel                59880    0.085548
Gaming              23784    0.033979
Gift Cards           9456    0.013509
Groceries          142578    0.203694
Healthcare          35889    0.051273
Online Retail      155097    0.221580
Restaurants        101360    0.144808
Streaming           47196    0.067427
Travel              28715    0.041024
Utilities           42027    0.060042
--------------------
col: card_type
             count  proportion
card_type                     
Amex         45797    0.065428
Discover     17235    0.024623
Mastercard  254857    0.364102
RuPay        37070    0.052960
Visa        345001    0.492887
--------------------
col: auth_method
                    count  proportion
auth_method                          
3D Secure          210036    0.300069
Biometric           93666    0.133816
No

## Feature Engineering - 1st Digit and 2 first Digit - Benford's Law

In [13]:
def extract_first_digit_features(df, numeric_cols, n=1):
    """Extracts the leading digits (e.g., 1st digit or first 2 digits)

    from numeric columns for Benford's Law analysis.

    Parameters:
    - df: pandas DataFrame
    - numeric_cols: list of columns to process
    - n_digits: number of leading digits to extract (default is 1)
    """
    df = df.copy()

    for col in numeric_cols:
        # 1. Convert to string, strip all non-digit characters (decimals, signs)
        str_series = (
            df[col].astype(str).str.replace(r"\D+", "", regex=True).str.lstrip("0")
        )

        # 2. Extract the first n digits (handling empty strings where values were 0)
        feature_name = (
            f"{col}_first_{n}_digit" if n > 1 else f"{col}_first_digit"
        )

        # Extract slice and handle cases where string length is less than n_digits
        df[feature_name] = (
            str_series.str[:n]
            .replace("", np.nan)
            .astype("float")  # Keeps NaN safe before int conversion if needed
            .fillna(-1)
            .astype(int)
        )

    return df

In [14]:
def extract_first_two_digits(df, numeric_cols):
  """Extracts the first two leading digits from numeric columns.

  Handles negatives, decimals, and leading zeros (e.g., 0.0034 -> 34).

  Parameters:
  - df: pandas DataFrame
  - numeric_cols: list of column names to process

  Returns:
  - DataFrame with new columns named '{col}_first_two_digits'
  """
  df = df.copy()

  for col in numeric_cols:
    # 1. Convert to string and remove non-digit characters (signs, decimals, exponents)
    # Then strip any leading zeros
    str_series = (
        df[col].astype(str).str.replace(r"[^\d]", "", regex=True).str.lstrip("0")
    )

    # 2. Extract the first 2 characters, handling empty strings (e.g., original value was 0)
    extracted = str_series.str[:2]

    # 3. Format into a clean numeric feature (using -1 for missing/zero values)
    feature_name = f"{col}_first_two_digits"
    df[feature_name] = (
        extracted.replace("", np.nan).astype("float").fillna(-1).astype(int)
    )

  return df

In [15]:
train[numeric_cols].head()

,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,time_of_day_hour,day_of_week,merchant_risk_score,prior_disputes,is_fraud
0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,17,3,29.2,0,0.0
1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,23,5,29.9,1,0.0
2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,15,0,24.1,0,0.0
3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,7,3,41.9,0,0.0
4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,4,4,48.0,0,0.0


In [17]:
numeric_features = ["amount_usd", "account_balance_usd"]
train_featured = extract_first_digit_features(train, numeric_features)
train_featured = extract_first_two_digits(train, numeric_features)

In [18]:
test_featured = extract_first_digit_features(test, numeric_features)
test_featured = extract_first_two_digits(test, numeric_features)

## Model using XGboost + TE + FE Digit

In [15]:
from sklearn.preprocessing import TargetEncoder

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [18]:
import xgboost as xgb

In [19]:
X = train.drop(columns=["id","is_fraud"]).copy()
y = train["is_fraud"].copy()

### Insert TE in the run_cv loop

In [20]:
def run_cv(fit_predict_fn, te_cols, name): # adding input te_cols for categorical variable required for TE
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        # splitting into training and evaluation
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y[tr_idx].copy(), y[va_idx].copy()
        # --- FIX: Fill NaNs and ensure string type for TE columns ---
        #for col in te_cols:
        #    X_tr.loc[:,col] = X_tr.loc[:,col].fillna("Missing").astype(str)
        #   X_va.loc[:,col] = X_va.loc[:,col].fillna("Missing").astype(str)
        #   X_test.loc[:,col] = X_test.loc[:,col].fillna("Missing").astype(str)
        
        # -----------------------------------------------------------
        # creating TE model for training data
        te = TargetEncoder(smooth="auto",target_type='binary')
        
        # fit on tr data
        te.fit(X_tr[te_cols],y_tr)

        # transform on tr data
        X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])

        # transform on va data
        X_va.loc[:, te_cols] = te.transform(X_va[te_cols])
        
        # transform on X_test data
        X_test = test.drop(columns=["id"]).copy()
        # X_test_obj = X_test[obj_cols].copy()
        X_test.loc[:, te_cols] = te.transform(X_test[obj_cols])

        # print(f'x_test',X_test.head(2))

        # print(f'X_tr_col: ',X_tr.info())
        for col in te_cols:
            X_tr[col] = X_tr[col].astype('float64')
            X_va[col] = X_va[col].astype('float64')
            X_test[col] = X_test[col].astype('float64')
            # X_tr.loc[:,col] = X_tr.loc[:,col].astype('float64')
            # X_va.loc[:,col] = X_va.loc[:,col].astype('float64')
            # X_test.loc[:,col] = X_test.loc[:,col].astype('float64')

        # print(f'x_tr ', X_tr.info())
        # print(f'x_va ', X_va.info())
        # print(f'x_test ', X_test.info())
        # start predit
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va,X_test)

        # save prediction in oof and in test
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [21]:
def fit_xgb(X_tr, y_tr, X_va, y_va,X_test):
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        n_estimators=10000,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20,
        colsample_bytree=0.8,
        subsample=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method='hist',
        enable_categorical=True,
        early_stopping_rounds=200,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [22]:
def save_sub(pred_1,pred_2, name_1,name_2):
    sub = submit.copy()
    sub['is_fraud'] = pred_1
    sub.to_csv(name_1, index=False)
    print('save:', name_1)

    oof_df = train[['id']].copy()
    oof_df.loc[:,name_2] = oof_xgb
    oof_df.to_csv(name_2,index=False)
    print('save:', name_2)

In [23]:
SEED = 2312
N_FOLDS = 5
TARGET = 'is_fraud'

    # Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [24]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, obj_cols, 'XGBoost')

  fold 0: AUC = 0.94979
  fold 1: AUC = 0.95152
  fold 2: AUC = 0.94416
  fold 3: AUC = 0.94430
  fold 4: AUC = 0.94229
XGBoost | OOF AUC = 0.94632 (mean 0.94641 ± 0.00358)


In [25]:
save_sub(test_xgb, oof_xgb, f'sub/sub_xgb_te_digitv2_{SEED}.csv', f'oof/oof_xgb_te_digitv2_{SEED}.csv')

save: sub/sub_xgb_te_digitv2_2312.csv
save: oof/oof_xgb_te_digitv2_2312.csv


In [25]:
seed = [42,67,1508,2312,106]

In [26]:
auc_xgb_dict = {}

In [28]:
for Seed in seed:
    SEED = Seed
    N_FOLDS = 5
    TARGET = 'is_fraud'

    # Stratified KFold cross validation
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    FOLDS = list(skf.split(X, y))

    oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, obj_cols, 'XGBoost')
    save_sub(test_xgb, oof_xgb, f'sub/sub_xgb_te_{SEED}.csv', f'oof/oof_xgb_te_{SEED}.csv')
    auc_xgb_dict[Seed] = auc_xgb

  fold 0: AUC = 0.94455
  fold 1: AUC = 0.94846
  fold 2: AUC = 0.95111
  fold 3: AUC = 0.94614
  fold 4: AUC = 0.94221
XGBoost | OOF AUC = 0.94646 (mean 0.94649 ± 0.00308)
save: sub/sub_xgb_te_42.csv
save: oof/oof_xgb_te_42.csv
  fold 0: AUC = 0.94719
  fold 1: AUC = 0.94695
  fold 2: AUC = 0.94497
  fold 3: AUC = 0.94703
  fold 4: AUC = 0.94649
XGBoost | OOF AUC = 0.94650 (mean 0.94653 ± 0.00081)
save: sub/sub_xgb_te_67.csv
save: oof/oof_xgb_te_67.csv
  fold 0: AUC = 0.94478
  fold 1: AUC = 0.94771
  fold 2: AUC = 0.94872
  fold 3: AUC = 0.94711
  fold 4: AUC = 0.94394
XGBoost | OOF AUC = 0.94642 (mean 0.94645 ± 0.00180)
save: sub/sub_xgb_te_1508.csv
save: oof/oof_xgb_te_1508.csv
  fold 0: AUC = 0.95012
  fold 1: AUC = 0.95131
  fold 2: AUC = 0.94461
  fold 3: AUC = 0.94382
  fold 4: AUC = 0.94205
XGBoost | OOF AUC = 0.94630 (mean 0.94638 ± 0.00365)
save: sub/sub_xgb_te_2312.csv
save: oof/oof_xgb_te_2312.csv
  fold 0: AUC = 0.94696
  fold 1: AUC = 0.94823
  fold 2: AUC = 0.94669
  fo